## Load GTFS Reference Data into TrainAnalysis KQL Database

Downloads Timetables For Realtime v2 GTFS bundle from Transport NSW,
extracts stops.txt, routes.txt, and stop_times.txt, and loads them
as reference tables into the TrainAnalysis KQL database.

Uses the **Kusto REST API** directly (no SDK needed) to avoid version
conflicts in the Fabric Spark environment.

**Run cadence:** Once initially, then re-run when timetables change.

In [ ]:
pip install requests pandas

In [ ]:
import requests
import pandas as pd
import zipfile
import io
import json
import time
from datetime import datetime

In [ ]:
# Transport NSW API key
myapikey = ""

# TrainAnalysis KQL database Query URI
# Find in Fabric portal: open TrainAnalysis Eventhouse -> copy Query URI
kusto_uri = ""

# KQL database name
database_name = "TrainAnalysis"

In [ ]:
# Authenticate - uses mssparkutils in Fabric, falls back to Azure CLI
try:
    token = mssparkutils.credentials.getToken("https://kusto.kusto.windows.net")
    print("Authenticated via Fabric notebook identity")
except NameError:
    import subprocess
    result = subprocess.run(
        ["az", "account", "get-access-token", "--resource",
         "https://kusto.kusto.windows.net", "--query", "accessToken", "-o", "tsv"],
        capture_output=True, text=True
    )
    token = result.stdout.strip()
    print("Authenticated via Azure CLI")

kusto_headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}

def kusto_query(query, is_mgmt=False):
    """Execute a KQL query or management command via REST API."""
    endpoint = "v1/rest/mgmt" if is_mgmt else "v2/rest/query"
    url = f"{kusto_uri}/{endpoint}"
    body = {"db": database_name, "csl": query}
    resp = requests.post(url, headers=kusto_headers, json=body, timeout=120)
    resp.raise_for_status()
    return resp.json()

# Test connectivity
result = kusto_query(".show database schema | count", is_mgmt=True)
print(f"Connected to {database_name} successfully")

In [ ]:
# Download GTFS static timetable bundle
url = "https://api.transport.nsw.gov.au/v1/gtfs/schedule/sydneytrains"
headers = {"Authorization": f"apikey {myapikey}", "Accept": "application/zip"}

print("Downloading GTFS bundle from Transport NSW...")
response = requests.get(url, headers=headers, timeout=120)
response.raise_for_status()

gtfs_zip = zipfile.ZipFile(io.BytesIO(response.content))
file_list = gtfs_zip.namelist()
print(f"Downloaded bundle with {len(file_list)} files:")
for name in sorted(file_list):
    info = gtfs_zip.getinfo(name)
    print(f"  {name} ({info.file_size:,} bytes)")

In [ ]:
def parse_gtfs_file(gtfs_zip, filename, usecols=None):
    with gtfs_zip.open(filename) as f:
        df = pd.read_csv(f, dtype=str, usecols=usecols)
    print(f"{filename}: {len(df):,} rows, columns: {list(df.columns)}")
    return df

df_stops = parse_gtfs_file(gtfs_zip, "stops.txt", usecols=[
    "stop_id", "stop_name", "stop_lat", "stop_lon",
    "location_type", "parent_station", "platform_code"
])
df_stops["stop_lat"] = pd.to_numeric(df_stops["stop_lat"], errors="coerce")
df_stops["stop_lon"] = pd.to_numeric(df_stops["stop_lon"], errors="coerce")

df_routes = parse_gtfs_file(gtfs_zip, "routes.txt", usecols=[
    "route_id", "route_short_name", "route_long_name",
    "route_type", "route_color", "route_text_color"
])

df_stop_times = parse_gtfs_file(gtfs_zip, "stop_times.txt", usecols=[
    "trip_id", "arrival_time", "departure_time",
    "stop_id", "stop_sequence"
])
df_stop_times["stop_sequence"] = pd.to_numeric(
    df_stop_times["stop_sequence"], errors="coerce"
).astype("Int64")

print(f"\nPreview - stops:\n{df_stops.head()}")
print(f"\nPreview - routes:\n{df_routes.head()}")

In [ ]:
# Create table schemas (idempotent)
table_commands = [
    ".create-merge table StopsReference (stop_id: string, stop_name: string, stop_lat: real, stop_lon: real, location_type: string, parent_station: string, platform_code: string)",
    ".create-merge table RoutesReference (route_id: string, route_short_name: string, route_long_name: string, route_type: string, route_color: string, route_text_color: string)",
    ".create-merge table StopTimesReference (trip_id: string, arrival_time: string, departure_time: string, stop_id: string, stop_sequence: int)"
]
for cmd in table_commands:
    kusto_query(cmd, is_mgmt=True)
print("Table schemas created/verified")

In [ ]:
def ingest_dataframe_via_rest(table_name, df, batch_size=5000):
    """Ingest a DataFrame into a KQL table using the streaming ingestion REST API."""
    total_rows = len(df)

    # Clear existing data
    print(f"\nClearing existing {table_name} data...")
    kusto_query(f".clear table {table_name} data", is_mgmt=True)

    # Streaming ingest endpoint
    ingest_url = f"{kusto_uri}/v1/rest/ingest/{database_name}/{table_name}?streamFormat=Csv"

    ingested = 0
    for start in range(0, total_rows, batch_size):
        batch = df.iloc[start:start + batch_size]
        csv_data = batch.to_csv(index=False, header=False)
        resp = requests.post(
            ingest_url,
            headers={"Authorization": f"Bearer {token}", "Content-Type": "text/csv"},
            data=csv_data.encode("utf-8"),
            timeout=120
        )
        resp.raise_for_status()
        ingested += len(batch)
        print(f"  {table_name}: {ingested:,}/{total_rows:,} rows ingested")

    # Verify
    time.sleep(2)
    result = kusto_query(f"{table_name} | count")
    frames = result if isinstance(result, list) else result.get("Tables", result.get("frames", []))
    for frame in frames:
        rows = frame.get("Rows", frame.get("data", []))
        if rows:
            count = rows[0][0] if isinstance(rows[0], list) else rows[0].get("Count", "?")
            print(f"  Verified: {table_name} has {count} rows")
            break
    return ingested

In [ ]:
print("=" * 60)
print("Ingesting reference data into TrainAnalysis KQL database")
print("=" * 60)

ingest_dataframe_via_rest("StopsReference", df_stops, batch_size=5000)
ingest_dataframe_via_rest("RoutesReference", df_routes, batch_size=5000)

print("\nIngesting StopTimesReference (this may take a few minutes)...")
ingest_dataframe_via_rest("StopTimesReference", df_stop_times, batch_size=10000)

print("\n" + "=" * 60)
print("Reference data load complete!")
print("=" * 60)

In [ ]:
# Verify reference data
test_queries = [
    ("Central Station stops", 'StopsReference | where stop_name contains "Central" | take 5'),
    ("Train routes", 'RoutesReference | take 10'),
    ("Sample stop times", 'StopTimesReference | take 5'),
]
for label, q in test_queries:
    print(f"\n--- {label} ---")
    result = kusto_query(q)
    frames = result if isinstance(result, list) else result.get("Tables", result.get("frames", []))
    for frame in frames:
        cols = [c["ColumnName"] for c in frame.get("Columns", frame.get("columns", []))]
        rows = frame.get("Rows", frame.get("data", []))
        if cols and rows:
            for row in rows:
                print(dict(zip(cols, row)))
            break